# Mixture-of-Experts — Paper-to-Code Mock (Colab)

**Papers:** Sparsely-Gated Mixture-of-Experts (Shazeer et al., 2017) — https://arxiv.org/abs/1701.06538 · Switch Transformers (Fedus et al., 2021) — https://arxiv.org/abs/2101.03961

Medium-hard mock (~30 min). Fill in the `MoE` stub, run the clustered toy task (router should specialize), then the sanity checks. Reference solution in the last cell.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. Implement the MoE layer (YOUR TASK)

Router → softmax → top-k → renormalize gates → dispatch each token to its k experts → weighted combine. Plus a Switch-style aux load-balancing loss: `N * sum_e f_e * p_e`.

In [ ]:
class MoE(nn.Module):
    def __init__(self, dim, hidden, n_experts, k=1):
        super().__init__()
        assert 1 <= k <= n_experts
        self.n_experts, self.k = n_experts, k
        self.router = nn.Linear(dim, n_experts)
        self.experts = nn.ModuleList(
            nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(), nn.Linear(hidden, dim))
            for _ in range(n_experts)
        )

    def forward(self, x):                          # x: (B, dim)
        # TODO: logits = self.router(x); probs = softmax over experts
        # TODO: topk_w, topk_idx = probs.topk(self.k); renormalize topk_w to sum to 1
        # TODO: for each slot/expert, dispatch the masked tokens and accumulate w * expert(x)
        # TODO: aux = self.load_balance_loss(probs, topk_idx)
        # return out, aux, probs, topk_idx
        raise NotImplementedError("fill me in")

    def load_balance_loss(self, probs, topk_idx):
        # TODO: frac = fraction of tokens routed to each expert (from topk_idx)
        # TODO: mean_prob = probs.mean over batch
        # TODO: return n_experts * sum(frac * mean_prob)
        raise NotImplementedError("fill me in")

## 2. Demonstrate the benefit (clustered toy task)
Several latent clusters, each with a DIFFERENT target map. A specializing router sends each cluster to its own expert — and each token only pays for k=1 expert.

In [ ]:
def make_clustered_data(n_per=400, dim=8, n_clusters=4, seed=0):
    g = torch.Generator().manual_seed(seed)
    centers = torch.randn(n_clusters, dim, generator=g) * 4.0
    maps = [torch.randn(dim, dim, generator=g) for _ in range(n_clusters)]
    X, Y, C = [], [], []
    for c in range(n_clusters):
        xc = centers[c] + 0.5 * torch.randn(n_per, dim, generator=g)
        yc = torch.tanh(xc @ maps[c])              # distinct nonlinear map per cluster
        X.append(xc); Y.append(yc); C.append(torch.full((n_per,), c))
    return torch.cat(X), torch.cat(Y), torch.cat(C)

torch.manual_seed(0)
dim, n_clusters = 8, 4
X, Y, C = make_clustered_data(dim=dim, n_clusters=n_clusters, seed=0)

moe = MoE(dim=dim, hidden=32, n_experts=n_clusters, k=1)
opt = torch.optim.Adam(moe.parameters(), lr=1e-2)
for _ in range(800):
    out, aux, _, _ = moe(X)
    loss = F.mse_loss(out, Y) + 0.01 * aux
    opt.zero_grad(); loss.backward(); opt.step()

moe.eval()
with torch.no_grad():
    _, _, _, topk_idx = moe(X)
assign = torch.zeros(n_clusters, moe.n_experts)
for cl in range(n_clusters):
    chosen = topk_idx[C == cl, 0]
    for e in range(moe.n_experts):
        assign[cl, e] = (chosen == e).float().mean()
print("cluster -> expert routing matrix:\n", assign)
print("dominant expert per cluster:", assign.argmax(dim=1).tolist())

total = sum(p.numel() for p in moe.parameters())
active = sum(p.numel() for p in moe.router.parameters()) + moe.k * sum(p.numel() for p in moe.experts[0].parameters())
print(f"TOTAL params {total}   ACTIVE/token {active}   ratio {active/total:.3f}")

## 3. Sanity checks

In [ ]:
m = MoE(dim=6, hidden=16, n_experts=5, k=2)
x = torch.randn(7, 6)
out, aux, probs, topk_idx = m(x)

# 1) top-k selects exactly k distinct experts per token
assert topk_idx.shape == (7, 2)
for row in topk_idx:
    assert len(set(row.tolist())) == 2

# 2) combine weights sum to 1 per token
tw, _ = probs.topk(m.k, dim=-1); tw = tw / tw.sum(dim=-1, keepdim=True)
assert torch.allclose(tw.sum(dim=-1), torch.ones(7), atol=1e-6)

# 3) output shape == input shape
assert out.shape == x.shape

# 4) aux loss lower when balanced than collapsed
N = m.n_experts
balanced = torch.full((100, N), 1.0 / N); bal_idx = torch.arange(100).remainder(N).unsqueeze(-1)
collapsed = torch.zeros(100, N); collapsed[:, 0] = 1.0; col_idx = torch.zeros(100, 1, dtype=torch.long)
assert m.load_balance_loss(balanced, bal_idx) < m.load_balance_loss(collapsed, col_idx)

# 5) active params/token < total; grad flows to router + selected experts
total = sum(p.numel() for p in m.parameters())
active = sum(p.numel() for p in m.router.parameters()) + m.k * sum(p.numel() for p in m.experts[0].parameters())
assert active < total
m.zero_grad(); o, a, _, ti = m(x); (o.sum() + a).backward()
assert m.router.weight.grad.abs().sum() > 0
for e in set(ti.flatten().tolist()):
    assert m.experts[e][0].weight.grad.abs().sum() > 0

# 6) router specialization (from the trained model above)
dominant = assign.argmax(dim=1)
assert len(set(dominant.tolist())) == n_clusters
for c in range(n_clusters):
    assert assign[c, dominant[c]] > 0.9

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
class MoESolution(nn.Module):
    def __init__(self, dim, hidden, n_experts, k=1):
        super().__init__()
        assert 1 <= k <= n_experts
        self.n_experts, self.k = n_experts, k
        self.router = nn.Linear(dim, n_experts)
        self.experts = nn.ModuleList(
            nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(), nn.Linear(hidden, dim))
            for _ in range(n_experts)
        )

    def forward(self, x):
        logits = self.router(x)
        probs = F.softmax(logits, dim=-1)
        topk_w, topk_idx = probs.topk(self.k, dim=-1)
        topk_w = topk_w / topk_w.sum(dim=-1, keepdim=True)
        out = torch.zeros_like(x)
        for slot in range(self.k):
            idx = topk_idx[:, slot]
            w = topk_w[:, slot].unsqueeze(-1)
            for e in range(self.n_experts):
                mask = idx == e
                if mask.any():
                    out[mask] += w[mask] * self.experts[e](x[mask])
        aux = self.load_balance_loss(probs, topk_idx)
        return out, aux, probs, topk_idx

    def load_balance_loss(self, probs, topk_idx):
        one_hot = F.one_hot(topk_idx, self.n_experts).sum(dim=1).clamp(max=1).float()
        frac = one_hot.mean(dim=0)
        mean_prob = probs.mean(dim=0)
        return self.n_experts * (frac * mean_prob).sum()